# ICM — Curiosity-driven Exploration by Self-supervised Prediction (2017)

_Papers · Reinforcement Learning_

**Reward the agent for being surprised: an intrinsic bonus equal to how badly a forward model predicts the next state's features, in a feature space that ignores what the agent cannot control.**

---

This notebook is a practice scaffold for the **ICM — Curiosity-driven Exploration by Self-supervised Prediction (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch (Colab)

In [ ]:
# torch is preinstalled in Colab -- nothing to pip-install.
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0); np.random.seed(0)

# --- 0. Sanity-check the lesson's worked example for the intrinsic reward (Eq. 6, eta=1). ---
ETA = 1.0
phi_hat = torch.tensor([0.2, 0.1])      # forward model's PREDICTED next features
phi_t1  = torch.tensor([1.0, 0.5])      # ACTUAL next features
r_int   = 0.5 * ETA * ((phi_hat - phi_t1) ** 2).sum()      # Eq. 6
print("worked example (novel):   error =", (phi_hat - phi_t1).tolist(),
      " r_int =", round(r_int.item(), 5))      # -> 0.4
phi_hat_learned = torch.tensor([0.98, 0.49])               # later, well-predicted
r_int_learned   = 0.5 * ETA * ((phi_hat_learned - phi_t1) ** 2).sum()
print("worked example (learned):  r_int =", round(r_int_learned.item(), 5))   # -> 0.00025


# --- 1. The toy SPARSE-REWARD chain: states 0..N; reward 1.0 ONLY at state N. ---
N = 12                       # long enough that random walking ~never reaches the end
N_ACT = 2                    # 0 = left, 1 = right

def step(s, a):
    s2 = max(0, min(N, s + (1 if a == 1 else -1)))
    r_ext = 1.0 if s2 == N else 0.0          # sparse: extrinsic reward only at the goal
    done  = (s2 == N)
    return s2, r_ext, done


# --- 2. The Intrinsic Curiosity Module (Track B: nn primitives). ---
class ICM(nn.Module):
    def __init__(self, n_states, n_act, feat=8):
        super().__init__()
        self.phi     = nn.Embedding(n_states, feat)                 # encoder phi(s)
        self.inverse = nn.Sequential(nn.Linear(2 * feat, 32), nn.ReLU(),
                                     nn.Linear(32, n_act))           # g: predict a_t  (Eq. 2)
        self.forward_= nn.Sequential(nn.Linear(feat + n_act, 32), nn.ReLU(),
                                     nn.Linear(32, feat))            # f: predict phi(s_{t+1})  (Eq. 4)
        self.n_act = n_act

    def features(self, s):
        return self.phi(torch.as_tensor(s, dtype=torch.long))

    def forward(self, s, a, s2):
        f_t, f_t1 = self.features(s), self.features(s2)
        a_oh = torch.nn.functional.one_hot(torch.as_tensor(a), self.n_act).float()
        # inverse model: predict the action from the two states (Eq. 2)
        a_logits = self.inverse(torch.cat([f_t, f_t1], dim=-1))
        # forward model: predict next features from current features + action (Eq. 4)
        f_t1_hat = self.forward_(torch.cat([f_t, a_oh], dim=-1))
        return a_logits, f_t1_hat, f_t1.detach()                    # detach target: r_int is a reward, not a phi-loss

icm   = ICM(N + 1, N_ACT)
opt   = torch.optim.Adam(icm.parameters(), lr=1e-2)
ce    = nn.CrossEntropyLoss()
BETA  = 0.2                  # weight: (1-beta) on inverse loss, beta on forward loss (Eq. 7)

def icm_step(s, a, s2):
    """Train the ICM on one transition and RETURN the intrinsic reward r_int (Eq. 6)."""
    a_logits, f_t1_hat, f_t1 = icm(s, a, s2)
    L_I = ce(a_logits.unsqueeze(0), torch.tensor([a]))             # inverse loss (Eq. 3) -> shapes phi
    L_F = 0.5 * ((f_t1_hat - f_t1) ** 2).sum()                     # forward loss (Eq. 5)
    loss = (1 - BETA) * L_I + BETA * L_F
    opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():                                          # intrinsic reward = forward error (Eq. 6)
        r_int = 0.5 * ETA * ((f_t1_hat - f_t1) ** 2).sum().item()
    return r_int


# --- 3. A tiny tabular Q-learning agent driven by r_ext + r_int. ---
def run(use_curiosity, episodes=400, max_steps=60, gamma=0.95, alpha=0.5, eps=0.1):
    global icm, opt
    torch.manual_seed(0); np.random.seed(0)
    icm = ICM(N + 1, N_ACT); opt = torch.optim.Adam(icm.parameters(), lr=1e-2)   # fresh ICM per run
    Q = np.zeros((N + 1, N_ACT))
    furthest_per_ep = []
    for ep in range(episodes):
        s, done, furthest = 0, False, 0
        for _ in range(max_steps):
            a = np.random.randint(N_ACT) if np.random.rand() < eps else int(Q[s].argmax())
            s2, r_ext, done = step(s, a)
            r_int = icm_step(s, a, s2) if use_curiosity else 0.0
            r = r_ext + r_int                                      # Eq. 7's reward sum
            Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a]) # tabular Q-learning update
            s = s2; furthest = max(furthest, s)
            if done: break
        furthest_per_ep.append(furthest)
    return furthest_per_ep

cur  = run(use_curiosity=True)
base = run(use_curiosity=False)     # ABLATION: extrinsic only (r_int = 0)

def reached(furthest, goal=N):  return sum(1 for f in furthest if f >= goal)
print(f"\nfurthest state reached, last 50 eps -- curiosity: {int(np.mean(cur[-50:]))} / {N}",
      f"  |  extrinsic-only: {int(np.mean(base[-50:]))} / {N}")
print(f"episodes that reached the goal (state {N}) -- curiosity: {reached(cur)}",
      f"  |  extrinsic-only: {reached(base)}")
# Curiosity drives the agent all the way to state N (it reaches the goal many times);
# the extrinsic-only ablation stalls near the start and essentially never reaches it.
# (Exact numbers vary by seed/hardware; our small run, not the paper's results.)

## Visualize the data & results

_On a sparse-reward chain (extrinsic reward ONLY at the far state N=12), does the ICM curiosity bonus drive the agent out to the goal, and does removing it (extrinsic reward only, same everything else) leave the agent stuck near the start? We train both and plot the furthest state reached per episode._

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Train two identical tabular Q-learners on the sparse-reward chain (reward only at N=12),
# differing ONLY in the reward they receive:
#
#   cur  = run(use_curiosity=True)    # green: reward = r_ext + r_int(Eq.6); reach grows to 12
#   base = run(use_curiosity=False)   # red:   reward = r_ext only; reach stays ~2-3 (stuck)
#
# Plotted: the furthest chain state reached in each episode.
# With curiosity, the ICM forward-model error pays a bonus for novel states, so the agent is
# pulled outward until it discovers the goal (and then exploits the extrinsic reward).
# Without it, no signal reaches the agent until it accidentally lands on state 12 -- which a
# random walk on a length-12 chain essentially never does -- so it never escapes the start.
# (Numbers are from our own run; the paper reports VizDoom / Super Mario exploration, not these.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked intrinsic reward. With $\eta = 1$, the forward model predicts
            $\hat{\phi}(s_{t+1}) = (0.2,\,0.1)$ but the true next features are $\phi(s_{t+1}) = (1.0,\,0.5)$.
            Compute the intrinsic reward $r^i_t$ (Eq. 6). Then a second transition is later predicted as
            $(0.98,\,0.49)$ against the same true $(1.0,\,0.5)$ — compute its $r^i_t$ and say what the drop
            means.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Error vector: $(0.2,0.1)-(1.0,0.5) = (-0.8,-0.4)$. — _Eq. 6 measures the gap between the forward model's prediction and the real next-state features._
- Squared L2 norm: $(-0.8)^2+(-0.4)^2 = 0.64+0.16 = 0.80$. — _$\lVert\cdot\rVert_2^2$ is the sum of squared components._
- Scale by $\eta/2 = 0.5$: $r^i_t = 0.5\times 0.80 = 0.40$. — _Eq. 6 halves the squared error and multiplies by $\eta$ ($=1$ here)._
- Second transition: error $(-0.02,-0.01)$, squared-sum $0.0005$, so $r^i_t = 0.5\times 0.0005 = 0.00025$. — _Once the model predicts this transition well, its error — and the bonus — collapse toward zero._

**Answer:** $r^i_t = 0.40$ for the surprising transition and $0.00025$ once it is learned. The bonus
                 falling almost to zero is the mechanism: a region pays a curiosity reward only while it is
                 still unpredictable, so the agent is continually pushed toward the unfamiliar frontier.
                 The notebook recomputes $0.5\times(0.64+0.16)=0.40$ and $0.5\times0.0005=0.00025$.

</details>

**Problem 2.** The ablation. Your agent reaches the far goal of the sparse-reward chain when it gets
            $r_t = r^e_t + r^i_t$. Now set the intrinsic reward to zero ($r^i_t = 0$, extrinsic only),
            keeping everything else (environment, agent, learning rate, seed) identical, and retrain. What
            happens to how far the agent explores, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the reward: use reward = r_extrinsic (delete the + r_int term); keep the agent, lr, episode count, and seed fixed. — _An honest ablation changes exactly one thing — the curiosity bonus — so any difference is attributable to it._
- Retrain and watch the furthest state reached: the no-curiosity agent stalls near the start (every early action looks equally worthless because $r^e=0$ there), while the ICM agent had pushed out to the goal. — _With sparse extrinsic reward, a pure reward-maximizer has no gradient until it accidentally hits the goal — which random walking essentially never does on a long chain._
- Conclude the curiosity bonus, not the agent or environment, is what drove the deep exploration. — _Both runs share everything except $r^i_t$; only the curious one reaches the goal, isolating curiosity as the cause._

**Answer:** The extrinsic-only agent barely leaves the starting region and (almost) never reaches the
                 far goal, while the ICM agent explores the whole chain and finds it. Since the only difference
                 is the intrinsic reward $r^i_t$ (Eq. 6), this isolates curiosity as the source of the
                 directed exploration: it supplies a learning signal precisely where the environment supplies
                 none. The CODEVIZ panel shows this contrast.

</details>

**Problem 3.** Imagine the chain has a "noisy" coordinate appended to each state that flips randomly every
            step and is completely unaffected by the agent's action. If the agent computed surprise on the
            raw state, what would go wrong — and how does ICM's inverse model prevent it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the random coordinate is unpredictable: a forward model on raw state can never predict it, so its error stays high forever. — _Irreducible noise produces permanent prediction error._
- That permanent error becomes a permanent intrinsic reward, so the agent fixates on the noise instead of exploring — the noisy-TV problem. — _Maximizing $r^i_t$ on raw observations rewards staring at randomness._
- ICM instead encodes states with $\phi$ trained by the inverse model: the random coordinate carries no information about which action was taken, so gradient descent on $L_I$ never encodes it, and $\phi$ drops it. — _A feature only survives if it helps predict the agent's action; uncontrollable noise does not._

**Answer:** On raw state the noise is forever unpredictable, so it pays a forever-positive curiosity
                 bonus and the agent gets stuck watching it (the noisy-TV failure). ICM avoids this because the
                 inverse model trains $\phi$ to keep only action-relevant information; the random coordinate
                 says nothing about the action, so $\phi$ ignores it, the forward model's error on it is zero,
                 and it contributes no bonus. This is precisely why the paper predicts in a learned feature
                 space rather than on raw observations.

</details>